# AF7 – Proyecto final
## Simulación de un robot manipulador planar de 2 GDL

Modelo dinámico obtenido mediante Euler–Lagrange:

\[
\tau=M(q)\ddot q+C(q,\dot q)\dot q+g(q)+f(\dot q)
\]

La simulación se realiza durante **10 segundos** y muestra la posición de las dos articulaciones.

> Nota: el enunciado no fija condiciones iniciales ni una función de torque. Para comprobar el modelo se usa un caso de prueba editable con condiciones iniciales en cero y torques sinusoidales.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Parámetros del robot
m1, l1, lc1, I1 = 23.902, 0.45, 0.091, 1.266
b1, fc1, fe1 = 2.288, 7.17, 8.8

m2, l2, lc2, I2 = 3.88, 0.45, 0.048, 0.093
b2, fc2, fe2 = 0.175, 1.734, 1.87

g = 9.81
h = m2*l1*lc2

def matriz_inercia(q):
    q1, q2 = q
    M11 = m1*lc1**2 + m2*l1**2 + m2*lc2**2 + 2*h*np.cos(q2) + I1 + I2
    M12 = m2*lc2**2 + h*np.cos(q2) + I2
    M22 = m2*lc2**2 + I2
    return np.array([[M11, M12], [M12, M22]])

def matriz_coriolis(q, qd):
    q1, q2 = q
    qd1, qd2 = qd
    s2 = np.sin(q2)
    return np.array([
        [-2*h*s2*qd2, -h*s2*qd2],
        [ h*s2*qd1,    0.0]
    ])

def vector_gravedad(q):
    q1, q2 = q
    G1 = g*((m1*lc1 + m2*l1)*np.sin(q1) + m2*lc2*np.sin(q1 + q2))
    G2 = g*m2*lc2*np.sin(q1 + q2)
    return np.array([G1, G2])

def vector_friccion(qd, tau, k=50.0):
    qd = np.asarray(qd)
    tau = np.asarray(tau)
    signo_suave = np.tanh(k*qd)

    viscosa = np.array([b1, b2])*qd
    coulomb = np.array([fc1, fc2])*signo_suave

    limite_estatico = np.array([fe1, fe2])
    tau_saturado = np.clip(tau, -limite_estatico, limite_estatico)
    estatica = (1.0 - np.abs(signo_suave))*tau_saturado

    return viscosa + coulomb + estatica

# Caso de prueba editable
def torque_motor(t):
    tau1 = 12.0*np.sin(0.8*t)
    tau2 = 3.0*np.sin(1.2*t)
    return np.array([tau1, tau2])

def modelo_robot(t, x):
    q = x[0:2]
    qd = x[2:4]

    tau = torque_motor(t)
    M = matriz_inercia(q)
    C = matriz_coriolis(q, qd)
    G = vector_gravedad(q)
    F = vector_friccion(qd, tau)

    qdd = np.linalg.solve(M, tau - C@qd - G - F)
    return np.concatenate((qd, qdd))

print("Modelo cargado correctamente.")


## Simulación

Condiciones iniciales del caso de prueba:

\[
q_1(0)=q_2(0)=\dot q_1(0)=\dot q_2(0)=0
\]

Intervalo:

\[
0\le t\le10\;s
\]


In [ ]:
x0 = [0.0, 0.0, 0.0, 0.0]

ti, tf = 0.0, 10.0
t_eval = np.linspace(ti, tf, 2001)

sol = solve_ivp(
    modelo_robot,
    [ti, tf],
    x0,
    t_eval=t_eval,
    method="RK45",
    max_step=0.02,
    rtol=1e-5,
    atol=1e-7
)

if not sol.success:
    raise RuntimeError(sol.message)

t = sol.t
q1, q2 = sol.y[0], sol.y[1]
qd1, qd2 = sol.y[2], sol.y[3]

q1_deg = np.rad2deg(q1)
q2_deg = np.rad2deg(q2)
qd1_deg = np.rad2deg(qd1)
qd2_deg = np.rad2deg(qd2)

print("Simulación terminada correctamente.")
print(f"Tiempo final: {t[-1]:.1f} s")


## Resultados

La primera gráfica cumple con el requisito de visualizar la posición de las dos articulaciones durante 10 segundos. La gráfica de velocidad se incluye como resultado complementario.


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(t, q1_deg, label="q1 - Articulación 1")
plt.plot(t, q2_deg, label="q2 - Articulación 2")
plt.xlabel("Tiempo [s]")
plt.ylabel("Posición angular [°]")
plt.title("Posición de las articulaciones del robot planar de 2 GDL")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10,5))
plt.plot(t, qd1_deg, label="dq1/dt")
plt.plot(t, qd2_deg, label="dq2/dt")
plt.xlabel("Tiempo [s]")
plt.ylabel("Velocidad angular [°/s]")
plt.title("Velocidad de las articulaciones del robot planar de 2 GDL")
plt.grid(True)
plt.legend()
plt.show()

print("Rangos obtenidos:")
print(f"q1: {q1_deg.min():.2f}° a {q1_deg.max():.2f}°")
print(f"q2: {q2_deg.min():.2f}° a {q2_deg.max():.2f}°")
